In [0]:
%sql
USE SCHEMA e_commerce;

##### Task 2
Profile the raw data.

What I want in the end:
- A summary showing row counts for each table
- A summary of null values by column
- A summary of distinct key counts for:
  - customer_id
  - item_id
  - order_id

In [0]:
%sql
-- A summary showing row counts for each table
SELECT
  (SELECT COUNT(*) FROM bronze_customers) AS customer_count,
  (SELECT COUNT(*) FROM bronze_items) AS item_count,
  (SELECT COUNT(*) FROM bronze_orders) AS order_count;


SELECT 'customers' AS table_name, COUNT(*) row_count FROM bronze_customers
UNION ALL
SELECT 'items' AS table_name, COUNT(*) row_count FROM bronze_items
UNION ALL
SELECT 'orders' AS table_name, COUNT(*) row_count FROM bronze_orders;


In [0]:
%sql
--A summary of null values by column
CREATE OR REPLACE TEMP VIEW bronze_customers_nulls AS
SELECT COUNT(*) total_rows,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) customer_id_nulls,
  SUM(CASE WHEN name IS NULL THEN 1 ELSE 0 END) name_nulls,
  SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) email_nulls,
  SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END) country_nulls,
  SUM(CASE WHEN created_at IS NULL THEN 1 ELSE 0 END) created_at_nulls
FROM bronze_customers;

CREATE OR REPLACE TEMP VIEW bronze_items_nulls AS
SELECT COUNT(*) total_rows,
  SUM(CASE WHEN item_id IS NULL THEN 1 ELSE 0 END) item_id_nulls,
  SUM(CASE WHEN product_name IS NULL THEN 1 ELSE 0 END) product_name_nulls,
  SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) price_nulls,
  SUM(CASE WHEN category IS NULL THEN 1 ELSE 0 END) category_nulls
FROM bronze_items;

CREATE OR REPLACE TEMP VIEW bronze_orders_nulls AS
SELECT COUNT(*) total_rows,
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) order_id_nulls,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) customer_id_nulls,
  SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) order_date_nulls,
  SUM(CASE WHEN status IS NULL THEN 1 ELSE 0 END) status_nulls
FROM bronze_orders;

CREATE OR REPLACE TEMP VIEW bronze_orders_items_nulls AS
SELECT
  SUM(CASE WHEN item.item_id IS NULL THEN 1 ELSE 0 END) item_id_nulls,
  SUM(CASE WHEN item.quantity IS NULL THEN 1 ELSE 0 END) quantity_nulls
FROM bronze_orders
LATERAL VIEW explode(items) AS item;

In [0]:
%sql
SELECT *
FROM (
  SELECT 'bronze_customers' AS table_name, 'customer_id' AS column_name, customer_id_nulls AS null_count FROM bronze_customers_nulls
  UNION ALL
  SELECT 'bronze_customers', 'name', name_nulls FROM bronze_customers_nulls
  UNION ALL
  SELECT 'bronze_customers', 'email', email_nulls FROM bronze_customers_nulls
  UNION ALL
  SELECT 'bronze_customers', 'country', country_nulls FROM bronze_customers_nulls
  UNION ALL
  SELECT 'bronze_customers', 'created_at', created_at_nulls FROM bronze_customers_nulls
  UNION ALL
  SELECT 'bronze_items', 'item_id', item_id_nulls FROM bronze_items_nulls
  UNION ALL
  SELECT 'bronze_items', 'product_name', product_name_nulls FROM bronze_items_nulls
  UNION ALL
  SELECT 'bronze_items', 'price', price_nulls FROM bronze_items_nulls
  UNION ALL
  SELECT 'bronze_items', 'category', category_nulls FROM bronze_items_nulls
  UNION ALL
  SELECT 'bronze_orders', 'order_id', order_id_nulls FROM bronze_orders_nulls
  UNION ALL
  SELECT 'bronze_orders', 'customer_id', customer_id_nulls FROM bronze_orders_nulls
  UNION ALL
  SELECT 'bronze_orders', 'order_date', order_date_nulls FROM bronze_orders_nulls
  UNION ALL
  SELECT 'bronze_orders', 'status', status_nulls FROM bronze_orders_nulls
  UNION ALL
  SELECT 'bronze_orders_items', 'item_id', item_id_nulls FROM bronze_orders_items_nulls
  UNION ALL
  SELECT 'bronze_orders_items', 'quantity', quantity_nulls FROM bronze_orders_items_nulls
) t
WHERE null_count > 0
ORDER BY table_name, null_count DESC, column_name

In [0]:
%sql
-- A summary of distinct key counts for:
SELECT
  (SELECT COUNT(DISTINCT(customer_id)) FROM bronze_customers) AS customer_count,
  (SELECT COUNT(DISTINCT(item_id)) FROM bronze_items) AS item_count,
  (SELECT COUNT(DISTINCT(order_id)) FROM bronze_orders) AS order_count;

SELECT COUNT(customer_id) AS total, COUNT(DISTINCT(customer_id)) AS distinct_count FROM bronze_customers
UNION ALL
SELECT COUNT(item_id) AS total, COUNT(DISTINCT(item_id)) AS distinct_count FROM bronze_items
UNION ALL
SELECT COUNT(order_id) AS total, COUNT(DISTINCT(order_id)) AS distinct_count FROM bronze_orders